In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"

# Folder containing extracted TalTech CSV files
RAW_DIR = DATA_DIR / "taltech_extracted"

# IFC model
IFC_FILE = DATA_DIR / "DS3_TalTech_V4.ifc"

# Output folders
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUT = OUTPUT_DIR / "01_extraction"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. STUDY PERIOD
# ============================================================

STUDY_START = pd.Timestamp(
    "2025-02-07 12:15:00",
    tz="UTC"
)

STUDY_END = pd.Timestamp(
    "2025-06-27 17:45:00",
    tz="UTC"
)

# ============================================================
# 3. VARIABLES RETAINED
# ============================================================

REQUIRED_TYPES = {
    "TEMPERATURESENSOR": "temperature",
    "HUMIDITYSENSOR": "relative_humidity",
    "CO2SENSOR": "co2",
}

# ============================================================
# 4. INPUT VERIFICATION
# ============================================================

print("Project directory :", PROJECT_DIR)
print("Data directory    :", DATA_DIR)
print("CSV folder        :", RAW_DIR)
print("IFC model         :", IFC_FILE)

assert RAW_DIR.exists(), (
    f"TalTech extracted folder not found: {RAW_DIR}"
)

assert IFC_FILE.exists(), (
    f"IFC file not found: {IFC_FILE}"
)

print("\n✓ TalTech extracted folder found")
print("✓ IFC model found")

print("\nStudy period:")
print(STUDY_START, "→", STUDY_END)

# ============================================================
# 5. FIND CSV FILES
# ============================================================

csv_files = sorted(RAW_DIR.rglob("*.csv"))

print("\nCSV files found:", len(csv_files))

assert len(csv_files) > 0, (
    f"No CSV files found in {RAW_DIR}"
)

print("\nFirst CSV files:")

for f in csv_files[:10]:
    print(" -", f.name)

# ============================================================
# 6. LOAD ALL CSV FILES
# ============================================================

frames = []

required_columns = {
    "device_id",
    "timestamp",
    "meas_type",
    "value",
    "unit"
}

for path in csv_files:

    try:
        df = pd.read_csv(path)

    except Exception as e:
        print(
            f"Could not read {path.name}: {e}"
        )
        continue

    missing_columns = (
        required_columns - set(df.columns)
    )

    if missing_columns:

        print(
            f"Skipped {path.name}. "
            f"Missing columns: {missing_columns}"
        )

        continue

    df = df[
        [
            "device_id",
            "timestamp",
            "meas_type",
            "value",
            "unit"
        ]
    ].copy()

    # Clean operational identifier
    df["device_id"] = (
        df["device_id"]
        .astype(str)
        .str.strip()
    )

    # Standardize measurement type
    df["meas_type"] = (
        df["meas_type"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Convert timestamps to UTC datetime
    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        utc=True,
        errors="coerce"
    )

    # Convert measurements to numeric
    df["value"] = pd.to_numeric(
        df["value"],
        errors="coerce"
    )

    # Preserve provenance
    df["source_file"] = path.name

    frames.append(df)

assert len(frames) > 0, (
    "No valid TalTech CSV file could be loaded."
)

raw = pd.concat(
    frames,
    ignore_index=True
)

print("\n====================================")
print("RAW DATA SUMMARY")
print("====================================")

print("Raw observations :", len(raw))
print(
    "Unique devices   :",
    raw["device_id"].nunique()
)

print("\nMeasurement types:")

print(
    raw["meas_type"]
    .value_counts()
)

# ============================================================
# 7. IDENTIFY DEVICES WITH T + RH + CO2
# ============================================================

relevant = raw[
    raw["meas_type"].isin(
        REQUIRED_TYPES.keys()
    )
].copy()

availability = (
    relevant
    .groupby(
        [
            "device_id",
            "meas_type"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

# Ensure expected columns exist
for measurement_type in REQUIRED_TYPES:

    if measurement_type not in availability.columns:

        availability[
            measurement_type
        ] = 0

eligible = availability[
    (
        availability["TEMPERATURESENSOR"] > 0
    )
    &
    (
        availability["HUMIDITYSENSOR"] > 0
    )
    &
    (
        availability["CO2SENSOR"] > 0
    )
].copy()

selected_devices = sorted(
    eligible.index.tolist()
)

print("\n====================================")
print("SELECTED DEVICES")
print("====================================")

print(
    "Devices with Temperature + RH + CO2:",
    len(selected_devices)
)

for device in selected_devices:

    print(" -", device)

# Save selected devices
selected_devices_df = pd.DataFrame(
    {
        "device_id": selected_devices
    }
)

selected_devices_df.to_csv(
    OUT / "01_selected_devices.csv",
    index=False
)

# ============================================================
# 8. FILTER STUDY VARIABLES + DEVICES + PERIOD
# ============================================================

selected = raw[
    raw["device_id"].isin(
        selected_devices
    )
    &
    raw["meas_type"].isin(
        REQUIRED_TYPES.keys()
    )
].copy()

selected["variable"] = (
    selected["meas_type"]
    .map(
        REQUIRED_TYPES
    )
)

selected = selected[
    selected["timestamp"]
    .between(
        STUDY_START,
        STUDY_END,
        inclusive="both"
    )
].copy()

selected = selected.sort_values(
    [
        "device_id",
        "variable",
        "timestamp"
    ]
).reset_index(
    drop=True
)

print("\n====================================")
print("FILTERED STUDY DATA")
print("====================================")

print(
    "Selected raw observations:",
    len(selected)
)

print(
    "Selected devices:",
    selected[
        "device_id"
    ].nunique()
)

print(
    "Variables:",
    selected[
        "variable"
    ].unique()
)

# ============================================================
# 9. BASIC DATA QUALITY CHECKS
# ============================================================

invalid_timestamps = (
    selected[
        "timestamp"
    ].isna().sum()
)

missing_values = (
    selected[
        "value"
    ].isna().sum()
)

duplicate_rows = (
    selected
    .duplicated(
        subset=[
            "device_id",
            "variable",
            "timestamp"
        ]
    )
    .sum()
)

print("\n====================================")
print("DATA QUALITY CHECK")
print("====================================")

print(
    "Invalid timestamps:",
    invalid_timestamps
)

print(
    "Missing numeric values:",
    missing_values
)

print(
    "Exact duplicate "
    "device-variable-timestamp rows:",
    duplicate_rows
)

# ============================================================
# 10. SERIES-LEVEL AUDIT
# ============================================================

audit = (
    selected
    .groupby(
        [
            "device_id",
            "variable"
        ]
    )
    .agg(
        observations=(
            "value",
            "size"
        ),
        valid_values=(
            "value",
            lambda x: x.notna().sum()
        ),
        first_timestamp=(
            "timestamp",
            "min"
        ),
        last_timestamp=(
            "timestamp",
            "max"
        ),
        unique_timestamps=(
            "timestamp",
            "nunique"
        )
    )
    .reset_index()
)

# ============================================================
# 11. MEDIAN TEMPORAL INTERVAL
# ============================================================

interval_rows = []

for (
    device_id,
    variable
), group in selected.groupby(
    [
        "device_id",
        "variable"
    ]
):

    timestamps = (
        group[
            "timestamp"
        ]
        .dropna()
        .sort_values()
    )

    intervals = (
        timestamps
        .diff()
        .dropna()
        .dt.total_seconds()
        / 60
    )

    median_interval = (
        intervals.median()
        if len(intervals) > 0
        else np.nan
    )

    interval_rows.append(
        {
            "device_id": device_id,
            "variable": variable,
            "median_interval_min":
                median_interval
        }
    )

interval_df = pd.DataFrame(
    interval_rows
)

audit = audit.merge(
    interval_df,
    on=[
        "device_id",
        "variable"
    ],
    how="left"
)

print("\n====================================")
print("SERIES AUDIT")
print("====================================")

print(
    audit.to_string(
        index=False
    )
)

# ============================================================
# 12. SAVE OUTPUTS
# ============================================================

selected.to_csv(
    OUT /
    "02_selected_raw_observations_long.csv",
    index=False
)

audit.to_csv(
    OUT /
    "03_series_audit.csv",
    index=False
)

# ============================================================
# 13. PROVENANCE MANIFEST
# ============================================================

manifest = pd.DataFrame(
    [
        {
            "field":
                "source_directory",
            "value":
                str(RAW_DIR)
        },
        {
            "field":
                "ifc_model",
            "value":
                str(IFC_FILE)
        },
        {
            "field":
                "study_start_utc",
            "value":
                str(STUDY_START)
        },
        {
            "field":
                "study_end_utc",
            "value":
                str(STUDY_END)
        },
        {
            "field":
                "selection_rule",
            "value":
                "device has native "
                "Temperature + RH + CO2 observations"
        },
        {
            "field":
                "selected_device_count",
            "value":
                len(selected_devices)
        },
        {
            "field":
                "interpolation_applied",
            "value":
                False
        },
        {
            "field":
                "raw_observations_preserved",
            "value":
                True
        }
    ]
)

manifest.to_csv(
    OUT /
    "04_extraction_provenance_manifest.csv",
    index=False
)

# ============================================================
# 14. FINAL SUMMARY
# ============================================================

print("\n====================================")
print("EXTRACTION COMPLETED")
print("====================================")

print(
    "Selected devices:",
    len(selected_devices)
)

print(
    "Selected variables:",
    3
)

print(
    "Selected observations:",
    len(selected)
)

print(
    "\nOutput folder:",
    OUT
)

print(
    "\nFiles created:"
)

for output_file in sorted(
    OUT.glob("*.csv")
):

    print(
        " -",
        output_file.name
    )